In [1]:
import pandas as pd
import time
from thor_requests.connect import Connect
from thor_requests.wallet import Wallet
from thor_requests.contract import Contract
import json
import os


In [3]:
# config 
RPC = "https://vethor-node-test.vechaindev.com" #testnet
#RPC = "" #mainnet
connector = Connect(RPC)
SMC_VESTING_ADDRESS = os.getenv('iPublicSaleVBVesting')

key_dict = {}
with open("../../keystore") as json_file:
  key_dict = json.load(json_file)
_owner = "0x23fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7"
_wallet = Wallet.fromKeyStore(ks=key_dict, password='passtest')

_contract = Contract.fromFile('../abi/PublicSaleVBVesting.json')

In [4]:
def check_beneficiary(address):
  try:
    res = connector.call(
    caller=_owner,
    contract=_contract, 
    func_name="getBeneficiary", 
    func_params=[address],
    to=SMC_VESTING_ADDRESS,
    )
    return res
  except:
    return None
  return None
    
# aaa = check_beneficiary("0x4B0897b0513fdC7C541B6d9D7E929C4e5364D2dB")
# print(aaa)
def add_beneficiary(address, amount):
  txn = connector.transact(
    wallet=_wallet,
    contract=_contract,
    func_name="addBeneficiary",
    func_params=[address,amount],
    to=SMC_VESTING_ADDRESS,
    
  )
  id = txn["id"]
  tx_id = connector.wait_for_tx_receipt(tx_id=id, timeout=20)
  return id,tx_id

# print(add_beneficiary("0x9a773a0c1710a5afd9d25eb5b0d2dca2239663e6",1300000000))
# print(connector.get_tx("0x314f4fbf3e680f01e594e72704cee7c5f0a88ce89420143a968cbc4e832a7609"))

def is_confirmed(tx_hash):
    try:
        receipt = connector.get_tx_receipt(tx_id=tx_hash)
        if str(receipt["reverted"]) == "False": ## transaction success => receipt["reverted"] == False  
            return True
        return False
    except:
        return False

# print(is_confirmed("0x9fedba3fab7b5953978f606ab6e6ca5e679e5c4683430279a1c93a6ee1d360e6"))


In [5]:
progress = []


In [18]:
# added = []

# Input the data file in csv format
data = pd.read_csv("data.csv")

for row in data.iterrows():
  address = row[1][0]
  amount = int(row[1][1])*10**18
  if amount <=0: continue
  check = check_beneficiary(address)
  if check is not None:
    if str(check['reverted']) == "False":
      print(f"beneficiary {address} is already existed, skip")
      # added.append({"address": address, "amount": amount, "status": "skip"})
      continue
  tx_id,tx_hash = add_beneficiary(address, amount)
  if tx_hash != "":
    print(f"adding beneficiary {address} , txhash: {tx_hash}")
    progress.append({"address": address, "amount": amount, "status": "added", "hash": tx_id})
  time.sleep(5)

beneficiary 0xE4A482E15Bd8D5cAEf13B2f0EfdE7Bf15B737929 is already existed, skip
beneficiary 0xaDF66a56f668Bd18C598af93207330273E62ccA8 is already existed, skip
beneficiary 0x4B0897b0513fdC7C541B6d9D7E929C4e5364D2dB is already existed, skip
beneficiary 0x817f3fb962b5b090356e953134f34e63c5a7a0ad is already existed, skip
beneficiary 0xfaae2dddb4afc844533f214273c0d89819d728e0 is already existed, skip
beneficiary 0xd4601ccac345484e84f8fed6fb02a40d13108c61 is already existed, skip
beneficiary 0x27b6b7245f0df46f5ce35658b2aeeb97d0f64912 is already existed, skip
beneficiary 0x8c330dc0f8b4039a4b4a9f386b3095f49d1dafa9 is already existed, skip
beneficiary 0x4dabdfc1d1304e99be2a5ecdb62b96ca58bdc16a is already existed, skip
beneficiary 0xb60b5208684ec09288f5e36b0d3caeceee343c82 is already existed, skip
adding beneficiary 0x9e2084495f5f4affd0d465a8a1c4f56d8fb1fc60 , txhash: {'gasUsed': 114780, 'gasPayer': '0x23fd7c63c35fd26ac5c4e7e6dc52cca6ab7511d7', 'paid': '0xfedce04c9318000', 'reward': '0x4c75767d

In [26]:
final = []
for item in progress:
  if "hash" in item:
    check = is_confirmed(item["hash"])
    item["confirm"] = check
    final.append(item)
df = pd.DataFrame(final)
df.amount = ((df.amount) / (10**18))
df.to_csv('../PublicVesting/save.csv')
df 
  

,address,amount,status,hash,confirm
0,0xE4A482E15Bd8D5cAEf13B2f0EfdE7Bf15B737929,1001.0,added,0xa53a7e2586114f2ac229f20f6881306d066c9ba137a6...,True
1,0xaDF66a56f668Bd18C598af93207330273E62ccA8,1002.0,added,0xcbeee16c0bee0d748f46661ea5977122a1882d69cfcd...,True
2,0x4B0897b0513fdC7C541B6d9D7E929C4e5364D2dB,1003.0,added,0xca3ae02e12f77b32a70ab2070d66a1769b0b3fdca6d5...,True
3,0x817f3fb962b5b090356e953134f34e63c5a7a0ad,1004.0,added,0xadd25e22c10b45b6d737def4992ea889af8954ad8310...,True
4,0xfaae2dddb4afc844533f214273c0d89819d728e0,1005.0,added,0x5d07fe33242d83df5c3ebb8f858a86b66f7a5c806823...,True
5,0xd4601ccac345484e84f8fed6fb02a40d13108c61,1006.0,added,0x8dacc2d0340d63168b9646c4d3f3b388394567937304...,True
6,0x27b6b7245f0df46f5ce35658b2aeeb97d0f64912,1007.0,added,0x366ca08712e08deafea69ba227766f3f953dba787fe1...,True
7,0x8c330dc0f8b4039a4b4a9f386b3095f49d1dafa9,1008.0,added,0xd14b88d233ee9106b0fc3618c261d0f8ccde9ea5ed9b...,True
8,0x4dabdfc1d1304e99be2a5ecdb62b96ca58bdc16a,1009.0,added,0x470c5d854abc8f95e7c5a0be2e19fa6f8cb572c88abb...,True
9,0xb60b5208684ec09288f5e36b0d3caeceee343c82,1010.0,added,0xf728bd0214f76f9d3bc1792f99f91067b033e7c9430a...,True
